# Gepard — tts inference demo

Runs the DPO checkpoint **`nineninesix/gepard-1.0`**.

The checkpoint is **self-describing**: it carries a `gepard_config.json`, so the
runner rebuilds the exact architecture (backbone, 32 audio heads, voice-cloning
compressor, `partial_rotary_factor`) from the checkpoint alone — no training
configs needed. Both repos are private, so authenticate first.

One runner — `GepardRunner`:
- default (`cfg_scale=1.0`) — plain single-pass generation
- `cfg_scale=2..3` — classifier-free guidance (better prosody/voice adherence, 2x compute on guided frames)

## Imports

In [ ]:
import torch
import librosa
from IPython.display import Audio as aplay

from gepard.inference import GepardRunner
from gepard.inference.codec_wrapper import UnfoldedCodecModel
from gepard.model.codec_ops import unfold_tokens

## Codec (tokens → waveform)

In [ ]:
codec_id = "nvidia/nemo-nano-codec-22khz-1.89kbps-21.5fps"
player = UnfoldedCodecModel.from_pretrained(codec_id)

## Load the model

One call — everything (architecture, special tokens, codec geometry, text
repetition policy) comes from the checkpoint's `gepard_config.json`. Expect a
clean load: no missing / unexpected keys.

In [ ]:
CHECKPOINT = "nineninesix/gepard-1.0"

model = GepardRunner.from_checkpoint(CHECKPOINT)

## Reference voice

Encode a reference clip into FSQ codes — the Q-Former compressor turns it into
speaker prefix tokens. Bundled demo voices in `assets/ref_audio/`:
`audio_en.wav`, `audio_ru.wav`, `linda.wav`, `nurisa_en.wav`, `ulan_emo.wav`.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

wave, _ = librosa.load("../assets/ref_audio/audio_ru.wav", sr=22050)
wave = torch.from_numpy(wave).unsqueeze(0).to(device)
wave_len = torch.tensor([wave.shape[-1]]).to(device)

with torch.inference_mode():
    tokens, _ = player.encode(audio=wave, audio_len=wave_len)

# fold codec groups out to the 32 per-dimension channels the model consumes
ref_codes = unfold_tokens(tokens.cpu(), num_levels=[8, 7, 6, 6]).permute(0, 2, 1).to(device)
ref_codes.shape  # [1, frames, 32]

torch.Size([1, 960, 32])

## Generate — plain runner

In [5]:
text = "Yesterday I decided I was going to be a responsible, organized person. I even said it out loud, which made it feel official."

tokens = model.generate(text, ref_codes=ref_codes, temperature=0.3)
# without a reference voice: tokens = model.generate(text, temperature=0.3)

In [6]:
def play(tokens):
    tokens = tokens.unsqueeze(0)
    enc_len = torch.tensor([tokens.shape[-1]]).to(device)
    with torch.inference_mode():
        audio, _ = player.decode_from_codes(tokens, enc_len)
    return aplay(audio.cpu().detach().flatten().numpy(), rate=22050)

play(tokens)

## Generate — classifier-free guidance

`cfg_scale=3` sharpens text/voice adherence; `cfg_frames=N` limits guidance to
the first N frames (`None` = the whole utterance). Short phrases are where CFG
helps the most.

In [7]:
text = "Hi there! Great to finally meet you."

tokens = model.generate(text, ref_codes=ref_codes, temperature=0.3,
                            cfg_scale=3, cfg_frames=None)
play(tokens)